In [ ]:
# Cell 1: Clone repo + install dependencies
!git clone https://github.com/Chocolatine75/miniWorldModel
%cd miniWorldModel
!pip install -r requirements.txt -q
print("Setup complete")

In [ ]:
# Cell 2: Collect rollouts
import yaml
from types import SimpleNamespace
from src.data import collect_random_rollouts

with open("config.yaml") as f:
    cfg = SimpleNamespace(**yaml.safe_load(f))
collect_random_rollouts(cfg)

In [ ]:
# Cell 3: Train the world model
import shutil, yaml
from types import SimpleNamespace
from src.train import train

with open("config.yaml") as f:
    cfg = SimpleNamespace(**yaml.safe_load(f))
train(cfg)
shutil.copy("loss_curves.png", "assets/loss_curves.png")

In [ ]:
# Cell 4: Plan + generate demo GIF
import torch, yaml
from pathlib import Path
from types import SimpleNamespace
from src.model import build_world_model
from src.env import make_env, get_obs, set_seed
from src.plan import run_mpc_episode
from src.viz import make_demo_gif

with open("config.yaml") as f:
    cfg = SimpleNamespace(**yaml.safe_load(f))

device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(cfg.seed)
Path("assets").mkdir(exist_ok=True)

model = build_world_model(cfg).to(device)
model.load_state_dict(torch.load(f"checkpoints/model_ep{cfg.n_epochs:03d}.pt", map_location=device, weights_only=True))
model.eval()

env = make_env(cfg.env_id, cfg.seed)
goal_raw, _ = env.reset(seed=cfg.seed + 99)
goal_obs = torch.tensor(get_obs(goal_raw, cfg.env_id))

frames = run_mpc_episode(model, env, goal_obs, cfg, device)
make_demo_gif(frames, "assets/demo.gif")
print(f"Demo: {len(frames)} frames")

In [ ]:
# Cell 5: Display demo GIF
from IPython.display import Image as IPImage
IPImage(filename="assets/demo.gif")

In [ ]:
# Cell 6: Display loss curves
from IPython.display import Image as IPImage
IPImage(filename="loss_curves.png")

In [ ]:
# Cell 7: Random vs CEM comparison GIF
import math
import torch
from src.env import make_env, get_obs, set_seed
from src.plan import run_mpc_episode
from src.viz import make_comparison_gif
from IPython.display import Image as IPImage

# Restore torch seed to match Cell 4 conditions
set_seed(cfg.seed)

# --- Goal observation: identical setup to Cell 4 ---
env_goal = make_env(cfg.env_id, cfg.seed)
goal_raw, _ = env_goal.reset(seed=cfg.seed + 99)
goal_obs = torch.tensor(get_obs(goal_raw, cfg.env_id))
env_goal.close()

# --- Random policy episode ---
# Raise the angle threshold to 90° so the pole falls completely before termination.
env_random = make_env(cfg.env_id, cfg.seed + 100)
env_random.unwrapped.theta_threshold_radians = math.pi / 2
env_random.unwrapped.x_threshold = 10.0
obs_raw, _ = env_random.reset()
frames_random = []
done = False
while not done:
    frames_random.append(env_random.render())
    action = env_random.action_space.sample()
    _, _, terminated, truncated, _ = env_random.step(action)
    done = terminated or truncated
frames_random.append(env_random.render())  # terminal frame (pole fully fallen)
env_random.close()

# --- CEM planner episode: same env seed as Cell 4 (guaranteed to work) ---
env_cem = make_env(cfg.env_id, cfg.seed)
frames_cem = run_mpc_episode(model, env_cem, goal_obs, cfg, device)
env_cem.close()

# Crop top white space (CartPole renders 600×400 but action is in the bottom ~300px)
frames_random = [f[80:, :] for f in frames_random]
frames_cem    = [f[80:, :] for f in frames_cem]

# --- Generate and display comparison GIF ---
make_comparison_gif(frames_random, frames_cem, "assets/comparison.gif")
print(f"Random episode: {len(frames_random)} frames")
print(f"CEM episode:    {len(frames_cem)} frames")
IPImage("assets/comparison.gif")

In [ ]:
# Cell 8: Download checkpoint
from IPython.display import FileLink
FileLink('checkpoints/model_ep020.pt')